# 🎯 **Use Case Reprojection Benchmark**

> ### 📌 **TL;DR**
>
> This notebook illustrates a use case of a reprojection for large rasters with and without using `dask`.
>
> Here we will compare the Reprojection feature with the following set-ups :
> - `rasterio`
> - `rioxarray`
> - `rioxarray + dask`

## **Dataset**

3 subsets with different sizes (100 MB, 1 GB and 6 GB) created from a Pléiades Neo image were used for this usecase.

- **100 MB** - (1 band, 1.7m resolution, 9104 x 5758 pixels)
- **1 GB** - (1 band, 0.53m resolution, 29135 x 18426 pixels)
- **6 GB** - (1 band, 0.22m resolution, 71367 x 45135 pixels)

## **Context**

The current CRS of our rasters is `EPSG:324643` and we will reproject them to `EPSG:4326`, in 3 different scenarios : using `rasterio`, `rioxarray` and finally `rioxarray`+`dask`.

The aim is to compare the time and memory consumed in each scenario and for images with different sizes, and see how it will affect final results.

For the purpose of this benchmark it was decided to keep only 1 band for each subset to mitigate biases linked to different reprojection techniques from each library.

The resampling method used for all reprojection scenarios is *`bilinear`*.

## **Hardware configuration**

- **CPU** : Intel Core i7-12700K @ 3.6 GHz
- **RAM** : 64 GB
- **OS** : Windows 11

In [1]:
from time import perf_counter
from memory_profiler import memory_usage # to compare mem usage
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import rioxarray as rxr

In [2]:
# dict to store source and dest file paths
rasters_io = {"SRC_PATH": [], "DST_RASTERIO": [], "DST_RIOXARRAY": [], "DST_DASK": []}
sizes = ["100mb", "1gb", "6gb"]

for size in sizes:
    rasters_io["SRC_PATH"].append(f"../data/rasters/img_{size}.tif")
    rasters_io["DST_RASTERIO"].append(f"../outputs/rasterio_out_benchmark_{size}.tif")
    rasters_io["DST_RIOXARRAY"].append(f"../outputs/rioxarray_out_benchmark_{size}.tif")
    rasters_io["DST_DASK"].append(f"../outputs/dask_out_benchmark_{size}.tif")

DST_CRS = "EPSG:4326" # target CRS for reprojection

In [3]:
# dict to display results for plots later
results = {
    "100mb": {"rasterio": {},
                "rioxarray": {},
                "rioxarray+dask_512": {},
                "rioxarray+dask_1024": {},
                "rioxarray+dask_auto": {}},
    "1gb":   {"rasterio": {},
                "rioxarray": {},
                "rioxarray+dask_1024": {},
                "rioxarray+dask_2048": {},
                "rioxarray+dask_auto": {}},
    "6gb":   {"rasterio": {},
                "rioxarray": {},
                "rioxarray+dask_2048": {},
                "rioxarray+dask_4096": {},
                "rioxarray+dask_auto": {}},
}

- ## **Reprojection with `rasterio`**

Here we will combine the reproject process and saving the new raster into a single function called `reproj_rasterio` :

In [4]:
def rasterio_reproject(src_path, dst_path):
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, 
            DST_CRS, 
            src.width, 
            src.height, 
            *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update(
            {
                "crs": DST_CRS,
                "transform": transform,
                "width": width,
                "height": height,
                "count": 1
            }
        )

        with rasterio.open(dst_path, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=DST_CRS,
                resampling=Resampling.bilinear,
            )

In [5]:
for size in sizes:
    print(f"\n--- RASTERIO {size} ---")
    t_start = perf_counter()
    mem_rasterio = memory_usage((rasterio_reproject, (rasters_io["SRC_PATH"][sizes.index(size)], rasters_io["DST_RASTERIO"][sizes.index(size)])), interval=2)
    t_end = perf_counter()

    elapsed_rasterio = round(t_end - t_start, 2)

    print(f"Duration : {elapsed_rasterio} seconds\nMax RAM: {max(mem_rasterio):.2f} MB\nRAM difference: {max(mem_rasterio) - min(mem_rasterio):.2f} MB\n")
    results[size]["rasterio"]["time"] = elapsed_rasterio # store time results
    results[size]["rasterio"]["peak_ram"] = round(max(mem_rasterio), 2) # store max RAM usage


--- RASTERIO 100mb ---
Duration : 6.74 seconds
Max RAM: 443.95 MB
RAM difference: 256.42 MB


--- RASTERIO 1gb ---
Duration : 30.18 seconds
Max RAM: 1981.31 MB
RAM difference: 1793.56 MB


--- RASTERIO 6gb ---
Duration : 175.71 seconds
Max RAM: 3510.77 MB
RAM difference: 3322.01 MB



- ## **Reprojection with `rioxarray`**

This time, we will reproject using `rioxarray`. We create a function called `reproj_rioxarray` :

In [6]:
def rioxarray_reproject(src_path, dst_path):
    with rxr.open_rasterio(src_path) as src:
        rxr_reprojected = src.rio.reproject(DST_CRS, resampling="bilinear")
        rxr_reprojected.sel(band=1).rio.to_raster(dst_path)

In [7]:
for size in sizes:
    print(f"\n--- RIOXARRAY {size} ---")
    t_start = perf_counter()
    mem_rioxarray = memory_usage((rioxarray_reproject, (rasters_io["SRC_PATH"][sizes.index(size)], rasters_io["DST_RIOXARRAY"][sizes.index(size)])), interval=2)
    t_end = perf_counter()

    elapsed_rioxarray = round(t_end - t_start, 2)

    print(f"Duration : {elapsed_rioxarray} seconds\nMax RAM: {max(mem_rioxarray):.2f} MB\nRAM difference: {max(mem_rioxarray) - min(mem_rioxarray):.2f} MB\n")
    results[size]["rioxarray"]["time"] = elapsed_rioxarray
    results[size]["rioxarray"]["peak_ram"] = round(max(mem_rioxarray), 2)


--- RIOXARRAY 100mb ---
Duration : 6.2 seconds
Max RAM: 831.41 MB
RAM difference: 442.50 MB


--- RIOXARRAY 1gb ---
Duration : 25.24 seconds
Max RAM: 5602.60 MB
RAM difference: 5018.36 MB


--- RIOXARRAY 6gb ---
Duration : 162.83 seconds
Max RAM: 32978.52 MB
RAM difference: 30398.49 MB



- ## **Reprojection with `rioxarray` + `dask`**

### **Testing chunk sizes for `dask`**

With `dask`, deciding the size of chunks is not an easy task, that can have an impact on final results. It is therefore important to choose wisely this parameter.

For this section, we will try different chunk sizes depending on each image :
- 100mb - `512`, `1024` & `"auto"`
- 1gb - `1024`, `2048` & `"auto"`
- 6gb - `2048`, `4096` & `"auto"`

In [8]:
# define chunk sizes to test for each image size
chunk_test_config = {
    "100mb": [512, 1024, "auto"],
    "1gb": [1024, 2048, "auto"],
    "6gb": [2048, 4096, "auto"]
}

for size in sizes:
    print(f"Image size : {size}")
    
    chunk_sizes = chunk_test_config[size]
    
    for chunk_size in chunk_sizes:
        print(f"\n--- Chunk size: {chunk_size} ---")

        def dask_reproject():
            chunks = {"x": chunk_size, "y": chunk_size} if chunk_size != "auto" else chunk_size
            with rxr.open_rasterio(rasters_io["SRC_PATH"][sizes.index(size)], chunks=chunks) as src:
                rxr_reprojected = src.rio.reproject(DST_CRS, resampling="bilinear")
                rxr_reprojected.sel(band=1).rio.to_raster(f"../outputs/test_chunk_{chunk_size}_{size}.tif")

        t_start = perf_counter()
        mem_dask = memory_usage(dask_reproject, interval=2)
        t_end = perf_counter()

        elapsed_dask = round(t_end - t_start, 2)
        peak_ram = round(max(mem_dask), 2)
        ram_diff = round(max(mem_dask) - min(mem_dask), 2)

        chunk_key = f"rioxarray+dask_{chunk_size}"
        results[size][chunk_key]["time"] = elapsed_dask
        results[size][chunk_key]["peak_ram"] = peak_ram

        print(f"Duration: {elapsed_dask} seconds")
        print(f"Peak RAM: {peak_ram} MB")
        print(f"RAM difference: {ram_diff} MB\n\n")

Image size : 100mb

--- Chunk size: 512 ---
Duration: 5.17 seconds
Peak RAM: 12180.61 MB
RAM difference: 5733.87 MB



--- Chunk size: 1024 ---
Duration: 6.3 seconds
Peak RAM: 6841.93 MB
RAM difference: 394.4 MB



--- Chunk size: auto ---
Duration: 6.3 seconds
Peak RAM: 7032.81 MB
RAM difference: 488.02 MB


Image size : 1gb

--- Chunk size: 1024 ---
Duration: 31.32 seconds
Peak RAM: 10279.26 MB
RAM difference: 3637.62 MB



--- Chunk size: 2048 ---
Duration: 25.45 seconds
Peak RAM: 10266.63 MB
RAM difference: 2937.74 MB



--- Chunk size: auto ---
Duration: 25.45 seconds
Peak RAM: 11241.21 MB
RAM difference: 3910.99 MB


Image size : 6gb

--- Chunk size: 2048 ---
Duration: 164.25 seconds
Peak RAM: 33008.41 MB
RAM difference: 24704.45 MB



--- Chunk size: 4096 ---
Duration: 164.14 seconds
Peak RAM: 33009.51 MB
RAM difference: 22443.71 MB



--- Chunk size: auto ---
Duration: 166.14 seconds
Peak RAM: 33028.21 MB
RAM difference: 22280.59 MB




## **Reprojection results**

In [9]:
import pandas as pd

# get data from results dict
rows = []
for size, methods in results.items():
    for method, stats in methods.items():
        rows.append({
            "size": size,
            "method": method,
            "time_sec": stats.get("time"),
            "peak_ram_mb": stats.get("peak_ram")
        })

# build pandas DF
df_results = pd.DataFrame(rows)

# custom sort order
method_order = {
    "rasterio": 0,
    "rioxarray": 1,
    "rioxarray+dask_512": 2,
    "rioxarray+dask_1024": 3,
    "rioxarray+dask_2048": 4,
    "rioxarray+dask_4096": 5,
    "rioxarray+dask_auto": 6,
}

df_results["method_sort_key"] = df_results["method"].map(method_order)
df_results = df_results.sort_values(["size", "method_sort_key"]).drop("method_sort_key", axis=1)
df_results = df_results.set_index(["size", "method"])

# better display with colors and formatting
df_styled = (
    df_results
    .style
    .background_gradient(cmap="YlOrRd", subset=["time_sec", "peak_ram_mb"])
    .format({"time_sec": "{:.2f}", "peak_ram_mb": "{:.1f}"})
    .set_caption("Reprojection Benchmark Results: rasterio vs rioxarray vs rioxarray+dask")
    .set_properties(**{"font-size": "13px", "padding": "12px", "text-align": "center", "border": "1px solid gray"})
)

display(df_styled)

## **Final Conclusion**

Based on the tests done across different image sizes (100MB, 1GB and 6GB) and with different scenarios (`rasterio`, `rioxarray`, `rioxarray+dask` (with different chunk sizes)), here are the conclusions :

#### **On chunk sizes**

- **Small Images (100MB)** : Chunk sizes seem to have an important impact on memory, as smaller chunks cause a significant spike in memory consumption, while the duration is almost the same for each scenarios.
- **Medium Images (1GB)** : In this case, chunk sizes seem to have a relative impact on memory usage; however processing times appear to decrease with larger chunks.
- **Large Images (6GB)** : Larger chunk sizes are preferable as they reduce the number of operations and improve overall flow rate.
- **Auto Chunking** : `dask`'s automatic chunking provides reasonable results but may not be optimal for all scenarios.

#### **On scenarios**

- `rasterio` : Fastest for simple reprojections, lowest memory usage, good baseline overall
- `rioxarray` : Convenient to use, slightly higher memory consumption compared to `rasterio`
- `rioxarray+dask` : Most flexible for large datasets, enables parallel processings, the chunk size is the key tuning parameter
<br>

## **Recommendations**

1. **Start with Auto Chunking** : Use `chunks="auto"` for initial tests to establish base performance.

2. **Tune Based on File Size** :
   - 100MB files: ~1024 chunks
   - 1GB files: ~2048 chunks  
   - 6GB+ files: ~4096 chunks

3. **Consider Memory Constraints** : In memory-limited environments, prefer smaller chunks (512-1024) to avoid out-of-memory errors.

4. **Profile Your Use Case** : Always run benchmarks on your specific data and hardware, as optimal chunk size depends on :
   - Available RAM
   - Dataset characteristics
   - Processing requirements